# 02 - Feature Engineering

Construccion de las variables que alimentan el clustering, a partir de los 30
meses de unidades vendidas (enero 2024 - junio 2026).

Tres bloques de variables:

| Bloque | Para que sirve |
|---|---|
| **Semestrales** | agregan la venta al ciclo de compra real del negocio |
| **Comportamiento** | describen *como* vende el producto: son las que agrupan |
| **Atributos** | describen *que es* el producto: enriquecen los perfiles |

Cada decision de calculo se justifica en su seccion. Son las decisiones que mas
influyen en el resultado final del modelo.

---

In [1]:
# Permite importar el paquete src desde la carpeta notebooks/
import sys
from pathlib import Path

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src import config as cfg
from src import utils

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)
utils.aplicar_estilo_graficas()

# En notebook queremos ver las figuras en linea, no solo guardarlas en disco
%matplotlib inline

print("Raiz del proyecto:", RAIZ)
print("Archivo fuente existe:", cfg.ARCHIVO_EXCEL.exists())

Raiz del proyecto: c:\Users\jaider.catano\OneDrive - Prebel S.A BIC\Documentos\Cosnova_Clustering\Cosnova_Clustering
Archivo fuente existe: True


In [2]:
from src import ingesta
from src import feature_engineering as fe

maestro = ingesta.cargar_maestro()
meses = [m for m in cfg.MESES if m in maestro.columns]
print(f"Maestro: {len(maestro):,} registros x {len(meses)} meses")

Maestro: 1,815 registros x 30 meses


In [3]:
maestro.head()

,Marca,EAN/UPC-Code,Status,Tipo Portafolio,Material,Descripcion,Categoria,Subcategoria,Grupo,Tono,Tipo,Caracteristica,PVP,Canal,ene-24,feb-24,mar-24,abr-24,may-24,jun-24,jul-24,ago-24,sep-24,oct-24,nov-24,dic-24,ene-25,feb-25,mar-25,abr-25,may-25,jun-25,jul-25,ago-25,sep-25,oct-25,nov-25,dic-25,ene-26,feb-26,mar-26,abr-26,may-26,jun-26,jul-26,ago-26
0,Essence,4059729337238,No change,New's junio 2026,PB0112297,"ESS. LAPIZ OJ. LONG-LASTING TN35 0,28G",Maquillaje,Ojos,Lapiz de ojos,Cafe claro,Lapiz,"Larga duración, resistente al agua",9900.0,Exito,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,4
1,Essence,4059729490292,No change,New's junio 2026,PB0112303,"ESS. BARRA BABY GOT BRONZE TN40 5,5G",Maquillaje,Rostro,Contorno,Cafe,Barra,Bronceador de rostro en barra,16500.0,Exito,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,3
2,Essence,4059729490971,No change,New's junio 2026,PB0112310,"ESS. PRIMER BARRA OJOS JELLY GRIP 4,6MLT",Maquillaje,Rostro,Primer,Transparente,Barra,Primer hidratante para contorno de ojos en barra,18900.0,Exito,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1
3,Essence,4059729511119,No change,New's junio 2026,PB0112321,"ESS. BALS. JUICY BOMB TN06 2,5G",Maquillaje,Labios,Balsamo,Transparente,Barra,Balsamo labial en barra transparente,17500.0,Exito,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
4,Essence,4059729517012,No change,New's junio 2026,PB0112322,ESS. ILUM. GOTAS DROP OF SUNSHINE 25MLT,Maquillaje,Rostro,Iluminador,Oro rosa,Liquido,Iluminador liquido,23500.0,Exito,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


## 1. Variables semestrales

El negocio compra dos veces al ano, asi que la unidad de analisis natural no es
el mes sino el semestre. Se agrupan los 30 meses en 5 semestres:

| Semestre | Meses | Estado |
|---|---|---|
| S1_2024 | ene-24 a jun-24 | completo |
| S2_2024 | jul-24 a dic-24 | completo |
| S1_2025 | ene-25 a jun-25 | completo |
| S2_2025 | jul-25 a dic-25 | completo |
| S1_2026 | ene-26 a jun-26 | completo (dato mas reciente) |
| **S2_2026** | jul-26 a dic-26 | **es lo que se va a planear** |

Por cada semestre se calculan tres cosas: unidades totales, meses con venta
(continuidad) y promedio mensual (base de la tendencia y la proyeccion).

In [ ]:
df = fe.agregar_variables_semestrales(maestro)

cols_sem = [c for c in df.columns if c.startswith(("unidades_S", "meses_con_venta_S", "promedio_S"))]
df[["Material", "Descripcion", "Canal"] + cols_sem].head(8)

In [ ]:
# Vista agregada: como evoluciona la venta semestre a semestre
resumen = pd.DataFrame({
    "unidades_totales": [df[f"unidades_{s}"].sum() for s in cfg.SEMESTRES],
    "registros_con_venta": [(df[f"unidades_{s}"] > 0).sum() for s in cfg.SEMESTRES],
    "meses_con_venta_prom": [df[f"meses_con_venta_{s}"].mean().round(2) for s in cfg.SEMESTRES],
}, index=list(cfg.SEMESTRES))
resumen["crecimiento_vs_anterior"] = resumen["unidades_totales"].pct_change().mul(100).round(1)
resumen

### Lectura

La venta crece semestre a semestre y el numero de registros con venta tambien:
el portafolio se esta ampliando, no solo vendiendo mas de lo mismo. Eso refuerza
la necesidad del clustering: cada semestre entran productos nuevos sin historia
propia para proyectar.

## 2. Variables de comportamiento

Estas son las 6 variables que efectivamente agrupan los productos. Cada una se
construye con una decision de calculo que vale la pena explicitar.

### 2.1 `promedio_mensual`: dividir por meses activos, no por 30

```
promedio_mensual = total_unidades / meses_activos
```

**Por que no dividir por 30.** Un producto lanzado en mayo de 2026 lleva 2 meses
en el mercado. Si se dividiera su venta por 30, su promedio saldria artificialmente
bajo y el modelo lo agruparia con los productos que llevan 30 meses vendiendo mal.
Son dos situaciones opuestas: uno es un lanzamiento, el otro un producto agotado
comercialmente.

Dividir por meses activos mide **intensidad de venta cuando el producto si estuvo
disponible**. La informacion de "cuanto tiempo lleva vendiendo" no se pierde:
la aporta `cobertura_meses`, que es una variable aparte.

In [ ]:
df["total_unidades"] = df[meses].sum(axis=1)
df["meses_activos"] = (df[meses] > 0).sum(axis=1)

# Comparacion de los dos criterios para dejar clara la diferencia
comparacion = df.loc[df["meses_activos"] > 0, ["Descripcion", "Canal", "total_unidades", "meses_activos"]].copy()
comparacion["prom_sobre_activos"] = (comparacion["total_unidades"] / comparacion["meses_activos"]).round(1)
comparacion["prom_sobre_30"] = (comparacion["total_unidades"] / 30).round(1)
comparacion["veces_mas_alto"] = (comparacion["prom_sobre_activos"] / comparacion["prom_sobre_30"]).round(1)

print("Productos con pocos meses activos: donde los dos criterios mas se separan")
comparacion.nsmallest(8, "meses_activos")[
    ["Descripcion", "Canal", "total_unidades", "meses_activos", "prom_sobre_activos", "prom_sobre_30", "veces_mas_alto"]
]

### 2.2 `cobertura_meses`: continuidad de la venta

```
cobertura_meses = 100 * meses_activos / 30
```

Es la variable mas legible del modelo y la que sostiene la regla de prioridad de
compra. Un producto con cobertura del 95% es un producto de rotacion permanente;
uno del 10% vende a picos aislados.

In [ ]:
df["promedio_mensual"] = np.where(df["meses_activos"] > 0,
                                  df["total_unidades"] / df["meses_activos"].replace(0, np.nan), 0)
df["promedio_mensual"] = df["promedio_mensual"].fillna(0).round(4)
df["cobertura_meses"] = (100 * df["meses_activos"] / len(meses)).round(2)

df[["cobertura_meses", "promedio_mensual", "total_unidades", "meses_activos"]].describe().round(2)

### 2.3 `tendencia`: crece o cae?

```
tendencia = promedio mensual de S2_2025 - promedio mensual de S1_2024
```

**Por que se comparan promedios y no totales.** Expresarlo en unidades por mes
hace la cifra comparable entre semestres de distinta longitud y facil de leer:
"este producto vende 12 unidades mas por mes que cuando arranco".

**Por que S2_2025 y no S1_2026.** S1_2026 es el dato mas reciente, pero comparar
un primer semestre contra un primer semestre mezclaria la tendencia con la
estacionalidad. S2_2025 es el ultimo semestre cerrado y S1_2024 el primero
observado: la diferencia entre ambos es tendencia de mediano plazo relativamente
limpia. S1_2026 se reserva para la proyeccion.

In [ ]:
df["tendencia"] = fe.calcular_tendencia(df)

print("Distribucion de la tendencia:")
print(df["tendencia"].describe().round(2).to_string())

con_venta = df[df["total_unidades"] > 0]
print(f"\nEn crecimiento: {(con_venta['tendencia'] > 0).sum():>5} registros")
print(f"En declive:     {(con_venta['tendencia'] < 0).sum():>5} registros")
print(f"Sin cambio:     {(con_venta['tendencia'] == 0).sum():>5} registros")

### 2.4 `estacionalidad`: en cual de las dos compras pedir?

```
estacionalidad = (S1_2024 + S1_2025) / (S2_2024 + S2_2025)
```

- **> 1**: vende mas en el primer semestre
- **= 1**: equilibrado
- **< 1**: vende mas en el segundo semestre

Es la variable de mayor valor operativo del modelo: dice literalmente en cual de
las dos compras del ano conviene pedir el producto.

**Dos decisiones de calculo:**

1. **Solo se usan 2024 y 2025**, los unicos anios con ambos semestres completos.
   Incluir 2026 (que solo tiene S1) inflaria artificialmente el numerador de todos
   los productos.

2. **El ratio se acota entre 0,33 y 3**. Si un producto vendio en S1 pero nunca en
   S2, el ratio matematico seria infinito. Un solo valor infinito destruye el
   escalado de la variable y con el la geometria del clustering completo. Acotar
   conserva la senal ("fuertemente S1") sin el efecto destructivo.

In [ ]:
df["estacionalidad"] = fe.calcular_estacionalidad(df)

# Interpretacion en tramos de negocio
tramos = pd.cut(df["estacionalidad"], bins=[0, 0.85, 1.15, 10],
                labels=["Vende mas en S2", "Equilibrado", "Vende mas en S1"])
print("Comportamiento estacional del portafolio:")
print(tramos.value_counts().to_string())

fig, ax = plt.subplots(figsize=(11, 4.5))
ax.hist(df.loc[df["total_unidades"] > 0, "estacionalidad"], bins=40,
        color="mediumpurple", edgecolor="white")
ax.axvline(1.0, color="black", linestyle="--", label="Equilibrio (S1 = S2)")
ax.set_title("Distribucion de la estacionalidad (solo registros con venta)")
ax.set_xlabel("Ratio S1 / S2")
ax.set_ylabel("Registros")
ax.legend()
plt.show()

### 2.5 `variabilidad`: que tan predecible es la venta?

```
variabilidad = desviacion estandar / media, calculada solo sobre meses con venta
```

Es el coeficiente de variacion. Un valor bajo indica venta estable y facil de
planear; un valor alto indica venta a picos, que exige mas colchon de inventario
para el mismo nivel de servicio.

**Por que solo sobre meses activos.** Si se incluyeran los ceros de un producto
nuevo, su variabilidad saldria altisima solo por no haber existido antes. Se
estaria midiendo antiguedad, no volatilidad.

In [ ]:
df["variabilidad"] = fe.calcular_variabilidad(df, meses)

print("Distribucion de la variabilidad (coeficiente de variacion):")
print(df.loc[df["total_unidades"] > 0, "variabilidad"].describe().round(3).to_string())

print("\nProductos mas estables (facil de planear, entre los de venta continua):")
estables = df[(df["cobertura_meses"] > 80) & (df["variabilidad"] > 0)]
print(estables.nsmallest(5, "variabilidad")[
    ["Descripcion", "Canal", "promedio_mensual", "variabilidad"]].to_string(index=False))

print("\nProductos mas volatiles (requieren colchon de inventario):")
print(df.nlargest(5, "variabilidad")[
    ["Descripcion", "Canal", "promedio_mensual", "variabilidad"]].to_string(index=False))

### 2.6 Variables de apoyo

- **`ingreso_estimado`** = `total_unidades` x `PVP`. **No entra al clustering**:
  como se vio en el EDA, es casi una transformacion lineal de `total_unidades` y
  agregarla contaria el volumen dos veces. Se conserva porque es la cifra que el
  negocio necesita para priorizar en pesos.
- **`mejor_semestre`**: en que semestre vendio mas. Enriquece el perfil y ayuda a
  identificar productos nuevos (su mejor semestre es el mas reciente).
- **`canal_cod`**: canal como 0/1/2, solo para filtros y reportes. **El modelo usa
  el One-Hot**, no este codigo: un codigo numerico le diria al algoritmo que
  Prosalon (2) esta "mas lejos" de Exito (0) que Farmatodo (1), lo cual no
  significa nada.
- **`sin_movimiento`**: marca los productos sin una sola unidad en 30 meses.

In [ ]:
df = fe.agregar_variables_comportamiento(df)

print("Distribucion de 'mejor_semestre':")
print(df["mejor_semestre"].value_counts().to_string())

## 3. Productos sin movimiento

Un 21% de los registros no vendio **ni una unidad** en 30 meses. Estos productos
se marcan y quedan **fuera del clustering principal** (cluster reservado -1).

**Por que excluirlos.** Todas sus variables valen exactamente cero, asi que
formarian un punto unico repetido cientos de veces. Eso hace dos danos: infla
artificialmente las metricas de calidad (un cluster de puntos identicos tiene
cohesion perfecta) y desplaza los centroides de los clusters reales.

No se pierden del entregable: aparecen en el Excel final con la etiqueta
"Sin Movimiento", y la cantidad a pedir la define el area comercial segun el plan
de lanzamiento, no el historico.

In [ ]:
sin_mov = df[df["sin_movimiento"]]
print(f"Registros sin movimiento: {len(sin_mov):,} ({utils.porcentaje(len(sin_mov), len(df))}%)")
print(f"Productos unicos afectados: {sin_mov['Material'].nunique()}")

print("\nPor canal:")
print(sin_mov["Canal"].value_counts().to_string())
print("\nPor status:")
print(sin_mov["Status"].value_counts().to_string())
print("\nPor tipo de portafolio (top 5):")
print(sin_mov["Tipo Portafolio"].value_counts().head(5).to_string())

### Lectura

Los productos sin movimiento se concentran en tipos de portafolio de novedades
recientes. Es decir: **en su mayoria no son fracasos comerciales sino productos
que todavia no han entrado a vender** en ese canal. Esa distincion importa
mucho para el comprador y es la razon por la que el cluster -1 se nombra
"Producto Nuevo Sin Historial" y no "Producto Descontinuado".

## 4. Codificacion One-Hot de los atributos

Los atributos de producto entran al modelo como variables dummy. Dos cuidados:

1. **Se descartan las columnas constantes.** `Categoria` vale `Maquillaje` en las
   1.815 filas: generaria una dummy de varianza cero que no aporta nada.

2. **Los niveles poco frecuentes se agrupan en `Otros`** (menos de 10 registros).
   Sin esto, los 60 valores de `Grupo` o los 50 de `Tono` crearian decenas de
   columnas casi vacias. En un espacio euclidiano eso equivale a ruido: cada
   columna casi vacia agrega una dimension donde practicamente todos los puntos
   valen 0 y unos pocos valen 1, lo que los vuelve outliers artificiales.

In [ ]:
dummies = fe.codificar_categoricas(df)
print(f"\nColumnas One-Hot generadas: {dummies.shape[1]}")
print("\nColumnas:")
for columna in dummies.columns:
    print(f"  {columna:<45} {int(dummies[columna].sum()):>5} registros")

## 5. Construccion final y validacion

Se ejecuta el paso completo y se valida el resultado.

In [ ]:
features = fe.ejecutar(maestro)
print(f"\nDataset de features: {features.shape[0]:,} filas x {features.shape[1]} columnas")

In [ ]:
# Validaciones de coherencia: si alguna falla, hay un error de calculo
meses_f = [m for m in cfg.MESES if m in features.columns]
cols_sem = [f"unidades_{s}" for s in cfg.SEMESTRES]

pruebas = {
    "La suma de semestres iguala el total de 30 meses":
        bool((features[cols_sem].sum(axis=1) == features["total_unidades"]).all()),
    "La cobertura esta entre 0 y 100":
        bool(features["cobertura_meses"].between(0, 100).all()),
    "La estacionalidad esta acotada en [0.33, 3]":
        bool(features["estacionalidad"].between(0.33, 3.0).all()),
    "No hay valores nulos en las features del modelo":
        bool(features[cfg.FEATURES_NUMERICAS].notna().all().all()),
    "Todo registro sin movimiento tiene total 0":
        bool((features.loc[features["sin_movimiento"], "total_unidades"] == 0).all()),
    "Todo registro con movimiento tiene promedio > 0":
        bool((features.loc[~features["sin_movimiento"], "promedio_mensual"] > 0).all()),
    "La variabilidad nunca es negativa":
        bool((features["variabilidad"] >= 0).all()),
}

for nombre, resultado in pruebas.items():
    print(f"  [{'OK ' if resultado else 'FALLO'}] {nombre}")

assert all(pruebas.values()), "Alguna validacion de features fallo"
print("\nTodas las validaciones pasaron.")

In [ ]:
# Estadisticas finales de las variables que entran al modelo
features[cfg.FEATURES_NUMERICAS].describe().T.round(3)

---

## Resumen de decisiones tomadas

| Decision | Razon |
|---|---|
| `promedio_mensual` sobre meses activos, no sobre 30 | no castigar lanzamientos recientes; la antiguedad ya la mide `cobertura_meses` |
| `tendencia` compara S2_2025 vs S1_2024 | evita mezclar tendencia con estacionalidad; deja S1_2026 libre para proyectar |
| `estacionalidad` solo con 2024 y 2025 | son los unicos anios con ambos semestres completos |
| `estacionalidad` acotada a [0,33 , 3] | un ratio infinito destruye el escalado y con el la geometria del clustering |
| `variabilidad` solo sobre meses activos | incluir ceros mediria antiguedad, no volatilidad |
| `ingreso_estimado` fuera del modelo | redundante con `total_unidades`; se conserva para el reporte |
| Canal por One-Hot, no por codigo 0/1/2 | un codigo impondria un orden y una distancia inexistentes entre canales |
| Niveles con menos de 10 registros a `Otros` | evita decenas de dimensiones casi vacias que actuan como ruido |
| Productos sin venta excluidos del clustering | son un punto identico repetido: inflan las metricas y desplazan centroides |

**Siguiente paso:** `03_Clustering.ipynb`.